# Build Silver & Gold Tables for Contracts, Combine, and Next Gen Stats

Run this notebook in Microsoft Fabric with the `lh_nfl` Lakehouse attached.

Prerequisites:

- The existing Silver/Gold model has been built (run `build_silver_gold_nfl_model.ipynb` first).
- Bronze tables exist from `import_contracts_combine_nextgen_to_bronze.ipynb`:
  - `bronze.nflverse_contracts`
  - `bronze.nflverse_combine`
  - `bronze.nflverse_nextgen_stats`

This notebook creates:

**Silver tables** (conformed, cleaned):
- `silver.contracts` — Player contracts with standardized keys
- `silver.combine` — NFL Combine results with standardized keys
- `silver.nextgen_stats` — Next Gen Stats with standardized keys

**Gold fact/dimension tables** (star schema, ready for analytics):
- `gold.fact_player_contract` — Contract facts joinable to dim_player, dim_team
- `gold.fact_combine` — Combine measurement facts joinable to dim_player
- `gold.fact_nextgen_passing` — NGS passing facts joinable to dim_player, dim_team, dim_season_week
- `gold.fact_nextgen_rushing` — NGS rushing facts joinable to dim_player, dim_team, dim_season_week
- `gold.fact_nextgen_receiving` — NGS receiving facts joinable to dim_player, dim_team, dim_season_week

All tables use the same `player_key` (gsis_id), `team_key` (team_abbr), and `season` conforming keys as the existing Gold model.

In [ ]:
from datetime import datetime, timezone

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
BRONZE_SCHEMA = "bronze"
WRITE_MODE = "overwrite"
RUN_UTC = datetime.now(timezone.utc).isoformat()

print(f"Notebook run UTC: {RUN_UTC}")


def run_sql(statement: str) -> None:
    print(statement.strip().splitlines()[0])
    spark.sql(statement)


run_sql(f"CREATE SCHEMA IF NOT EXISTS `{SILVER_SCHEMA}`")
run_sql(f"CREATE SCHEMA IF NOT EXISTS `{GOLD_SCHEMA}`")

## Silver: Contracts

Conforms OverTheCap contract data. The bronze table has a nested `cols` column (list of structs) containing per-year cap breakdowns. We explode it to get one row per player-team-season. The table already includes `gsis_id` which maps directly to `player_key`.

In [ ]:
run_sql(
    f"""
    CREATE OR REPLACE TABLE {SILVER_SCHEMA}.contracts
    USING DELTA
    AS
    WITH exploded AS (
        SELECT
            c.player AS player_name,
            c.gsis_id AS player_key,
            c.position,
            c.is_active,
            CAST(c.year_signed AS INT) AS year_signed,
            CAST(c.years AS INT) AS contract_years,
            CAST(c.value AS DOUBLE) AS total_value,
            CAST(c.apy AS DOUBLE) AS avg_per_year,
            CAST(c.guaranteed AS DOUBLE) AS guaranteed,
            CAST(c.apy_cap_pct AS DOUBLE) AS apy_cap_pct,
            CAST(c.inflated_value AS DOUBLE) AS inflated_value,
            CAST(c.inflated_apy AS DOUBLE) AS inflated_apy,
            CAST(c.inflated_guaranteed AS DOUBLE) AS inflated_guaranteed,
            c.otc_id,
            EXPLODE(c.cols) AS yr
        FROM {BRONZE_SCHEMA}.nflverse_contracts c
        WHERE c.player IS NOT NULL
          AND c.cols IS NOT NULL
    )
    SELECT
        player_name,
        player_key,
        yr.team AS team_key,
        position,
        CAST(yr.year AS INT) AS season,
        year_signed,
        contract_years,
        total_value,
        avg_per_year,
        guaranteed,
        apy_cap_pct,
        inflated_value,
        inflated_apy,
        inflated_guaranteed,
        CAST(yr.cap_number AS DOUBLE) AS cap_number,
        CAST(yr.cap_percent AS DOUBLE) AS cap_pct,
        CAST(yr.cash_paid AS DOUBLE) AS cash_paid,
        CAST(yr.base_salary AS DOUBLE) AS base_salary,
        CAST(yr.prorated_bonus AS DOUBLE) AS prorated_bonus,
        CAST(yr.roster_bonus AS DOUBLE) AS roster_bonus,
        CAST(yr.option_bonus AS DOUBLE) AS option_bonus,
        CAST(yr.other_bonus AS DOUBLE) AS other_bonus,
        CAST(yr.workout_bonus AS DOUBLE) AS workout_bonus,
        CAST(yr.guaranteed_salary AS DOUBLE) AS guaranteed_salary,
        CAST(yr.per_game_roster_bonus AS DOUBLE) AS per_game_roster_bonus,
        is_active,
        otc_id
    FROM exploded
    WHERE yr.year IS NOT NULL
    """
)

## Silver: Combine

NFL Combine results with standardized player key. The combine data includes `pfr_id` which we can use to join. We also join on name + draft year as a fallback.

In [ ]:
run_sql(
    f"""
    CREATE OR REPLACE TABLE {SILVER_SCHEMA}.combine
    USING DELTA
    AS
    SELECT
        CAST(c.season AS INT) AS draft_season,
        c.player_name,
        p.player_key,
        c.pos AS position,
        c.school,
        CAST(c.ht AS DOUBLE) AS height_inches,
        CAST(c.wt AS DOUBLE) AS weight_lbs,
        CAST(c.forty AS DOUBLE) AS forty_yard_dash,
        CAST(c.vertical AS DOUBLE) AS vertical_jump,
        CAST(c.bench AS INT) AS bench_press_reps,
        CAST(c.broad_jump AS DOUBLE) AS broad_jump,
        CAST(c.cone AS DOUBLE) AS three_cone_drill,
        CAST(c.shuttle AS DOUBLE) AS shuttle,
        CAST(c.draft_ovr AS INT) AS draft_overall_pick,
        CAST(c.draft_round AS INT) AS draft_round,
        c.draft_team AS team_key,
        c.pfr_id,
        c.cfb_id
    FROM {BRONZE_SCHEMA}.nflverse_combine c
    LEFT JOIN {SILVER_SCHEMA}.players p
        ON LOWER(TRIM(c.player_name)) = LOWER(TRIM(p.display_name))
       AND CAST(c.season AS INT) = p.rookie_season
    WHERE c.player_name IS NOT NULL
    """
)

## Silver: Next Gen Stats

Next Gen Stats come in three stat types (passing, rushing, receiving) that were loaded into a single bronze table. Columns not present in a given stat type are NULL. We keep them unified in silver with a `stat_type` discriminator derived from the source file path.

In [ ]:
run_sql(
    f"""
    CREATE OR REPLACE TABLE {SILVER_SCHEMA}.nextgen_stats
    USING DELTA
    AS
    SELECT
        n.player_gsis_id AS player_key,
        n.player_display_name AS player_name,
        n.team_abbr AS team_key,
        n.player_position AS position,
        CAST(n.season AS INT) AS season,
        CAST(n.week AS INT) AS week,
        n.season_type,
        CASE
            WHEN n._source_file_path LIKE '%passing%' THEN 'passing'
            WHEN n._source_file_path LIKE '%rushing%' THEN 'rushing'
            WHEN n._source_file_path LIKE '%receiving%' THEN 'receiving'
            ELSE 'unknown'
        END AS stat_type,
        -- Passing metrics
        CAST(n.avg_time_to_throw AS DOUBLE) AS avg_time_to_throw,
        CAST(n.avg_completed_air_yards AS DOUBLE) AS avg_completed_air_yards,
        CAST(n.avg_intended_air_yards AS DOUBLE) AS avg_intended_air_yards,
        CAST(n.avg_air_yards_differential AS DOUBLE) AS avg_air_yards_differential,
        CAST(n.aggressiveness AS DOUBLE) AS aggressiveness,
        CAST(n.max_completed_air_distance AS DOUBLE) AS max_completed_air_distance,
        CAST(n.avg_air_yards_to_sticks AS DOUBLE) AS avg_air_yards_to_sticks,
        CAST(n.passer_rating AS DOUBLE) AS passer_rating,
        CAST(n.completions AS INT) AS completions,
        CAST(n.attempts AS INT) AS attempts,
        CAST(n.pass_yards AS DOUBLE) AS pass_yards,
        CAST(n.pass_touchdowns AS INT) AS pass_touchdowns,
        CAST(n.interceptions AS INT) AS interceptions,
        CAST(n.expected_completion_percentage AS DOUBLE) AS expected_completion_percentage,
        CAST(n.completion_percentage_above_expectation AS DOUBLE) AS completion_pct_above_expectation,
        CAST(n.completion_percentage AS DOUBLE) AS completion_percentage,
        CAST(n.max_air_distance AS DOUBLE) AS max_air_distance,
        CAST(n.avg_air_distance AS DOUBLE) AS avg_air_distance,
        -- Rushing metrics
        CAST(n.efficiency AS DOUBLE) AS rush_efficiency,
        CAST(n.percent_attempts_gte_eight_defenders AS DOUBLE) AS pct_attempts_8plus_defenders,
        CAST(n.avg_time_to_los AS DOUBLE) AS avg_time_to_los,
        CAST(n.rush_attempts AS INT) AS rush_attempts,
        CAST(n.rush_yards AS DOUBLE) AS rush_yards,
        CAST(n.avg_rush_yards AS DOUBLE) AS avg_rush_yards,
        CAST(n.rush_touchdowns AS INT) AS rush_touchdowns,
        CAST(n.expected_rush_yards AS DOUBLE) AS expected_rush_yards,
        CAST(n.rush_yards_over_expected AS DOUBLE) AS rush_yards_over_expected,
        CAST(n.rush_yards_over_expected_per_att AS DOUBLE) AS rush_yards_over_expected_per_att,
        CAST(n.rush_pct_over_expected AS DOUBLE) AS rush_pct_over_expected,
        -- Receiving metrics
        CAST(n.avg_cushion AS DOUBLE) AS avg_cushion,
        CAST(n.avg_separation AS DOUBLE) AS avg_separation,
        CAST(n.percent_share_of_intended_air_yards AS DOUBLE) AS pct_share_intended_air_yards,
        CAST(n.receptions AS INT) AS receptions,
        CAST(n.targets AS INT) AS targets,
        CAST(n.catch_percentage AS DOUBLE) AS catch_percentage,
        CAST(n.yards AS DOUBLE) AS receiving_yards,
        CAST(n.avg_yac AS DOUBLE) AS avg_yac,
        CAST(n.avg_expected_yac AS DOUBLE) AS avg_expected_yac,
        CAST(n.avg_yac_above_expectation AS DOUBLE) AS avg_yac_above_expectation,
        CAST(n.rec_touchdowns AS INT) AS rec_touchdowns
    FROM {BRONZE_SCHEMA}.nflverse_nextgen_stats n
    WHERE n.player_gsis_id IS NOT NULL
    """
)

## Gold: fact_player_contract

One row per player-team-season contract year. Joinable to `dim_player` (via player_key) and `dim_team` (via team_key). Enables analysis of cap allocation efficiency correlated with performance.

In [ ]:
run_sql(
    f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_player_contract
    USING DELTA
    AS
    SELECT
        player_key,
        player_name,
        team_key,
        position,
        season,
        year_signed,
        contract_years,
        total_value,
        avg_per_year,
        guaranteed,
        apy_cap_pct,
        inflated_value,
        inflated_apy,
        inflated_guaranteed,
        cap_number,
        cap_pct,
        cash_paid,
        base_salary,
        prorated_bonus,
        roster_bonus,
        option_bonus,
        other_bonus,
        workout_bonus,
        guaranteed_salary,
        per_game_roster_bonus,
        is_active,
        otc_id
    FROM {SILVER_SCHEMA}.contracts
    WHERE season IS NOT NULL
    """
)

## Gold: fact_combine

One row per player combine appearance. Joinable to `dim_player` (via player_key). Enables prospect athletic profiling and correlation with career performance.

In [ ]:
run_sql(
    f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_combine
    USING DELTA
    AS
    SELECT
        player_key,
        player_name,
        team_key,
        position,
        draft_season,
        school,
        height_inches,
        weight_lbs,
        forty_yard_dash,
        vertical_jump,
        bench_press_reps,
        broad_jump,
        three_cone_drill,
        shuttle,
        draft_overall_pick,
        draft_round,
        draft_pick,
        pfr_id,
        cfb_id
    FROM {SILVER_SCHEMA}.combine
    WHERE draft_season IS NOT NULL
    """
)

## Gold: Next Gen Stats Facts

Split into three fact tables by stat_type for clean star-schema usage. Each is joinable to `dim_player`, `dim_team`, and `dim_season_week`.

In [ ]:
run_sql(
    f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_nextgen_passing
    USING DELTA
    AS
    SELECT
        player_key,
        player_name,
        team_key,
        position,
        season,
        week,
        season_type,
        CONCAT(CAST(season AS STRING), '_', LPAD(CAST(week AS STRING), 2, '0')) AS season_week_key,
        avg_time_to_throw,
        avg_completed_air_yards,
        avg_intended_air_yards,
        avg_air_yards_differential,
        aggressiveness,
        max_completed_air_distance,
        avg_air_yards_to_sticks,
        passer_rating,
        completions,
        attempts,
        pass_yards,
        pass_touchdowns,
        interceptions,
        expected_completion_percentage,
        completion_pct_above_expectation,
        completion_percentage,
        max_air_distance
    FROM {SILVER_SCHEMA}.nextgen_stats
    WHERE stat_type = 'passing'
    """
)

run_sql(
    f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_nextgen_rushing
    USING DELTA
    AS
    SELECT
        player_key,
        player_name,
        team_key,
        position,
        season,
        week,
        season_type,
        CONCAT(CAST(season AS STRING), '_', LPAD(CAST(week AS STRING), 2, '0')) AS season_week_key,
        rush_efficiency,
        pct_attempts_8plus_defenders,
        avg_time_to_los,
        rush_attempts,
        rush_yards,
        avg_rush_yards,
        rush_touchdowns,
        expected_rush_yards,
        rush_yards_over_expected,
        rush_yards_over_expected_per_att,
        rush_pct_over_expected
    FROM {SILVER_SCHEMA}.nextgen_stats
    WHERE stat_type = 'rushing'
    """
)

run_sql(
    f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_nextgen_receiving
    USING DELTA
    AS
    SELECT
        player_key,
        player_name,
        team_key,
        position,
        season,
        week,
        season_type,
        CONCAT(CAST(season AS STRING), '_', LPAD(CAST(week AS STRING), 2, '0')) AS season_week_key,
        avg_cushion,
        avg_separation,
        recv_avg_intended_air_yards,
        pct_share_intended_air_yards,
        receptions,
        targets,
        catch_percentage,
        avg_yac,
        avg_expected_yac,
        avg_yac_above_expectation,
        rec_touchdowns
    FROM {SILVER_SCHEMA}.nextgen_stats
    WHERE stat_type = 'receiving'
    """
)

## Summary

Display row counts for all tables created by this notebook.

In [ ]:
new_tables = [
    (SILVER_SCHEMA, "contracts"),
    (SILVER_SCHEMA, "combine"),
    (SILVER_SCHEMA, "nextgen_stats"),
    (GOLD_SCHEMA, "fact_player_contract"),
    (GOLD_SCHEMA, "fact_combine"),
    (GOLD_SCHEMA, "fact_nextgen_passing"),
    (GOLD_SCHEMA, "fact_nextgen_rushing"),
    (GOLD_SCHEMA, "fact_nextgen_receiving"),
]

results = []
for schema, table in new_tables:
    full_name = f"`{schema}`.`{table}`"
    row_count = spark.table(full_name).count()
    col_count = len(spark.table(full_name).columns)
    results.append({"schema": schema, "table": table, "rows": row_count, "columns": col_count})
    print(f"{full_name}: {row_count:,} rows, {col_count} columns")

print(f"\n✅ All {len(results)} tables created successfully.")

## Relationship Reference

These new Gold tables conform to the existing dimensional model:

```
fact_player_contract.player_key  → dim_player.player_key
fact_player_contract.team_key    → dim_team.team_key
fact_player_contract.season      → dim_season_week.season

fact_combine.player_key          → dim_player.player_key
fact_combine.team_key            → dim_team.team_key

fact_nextgen_passing.player_key      → dim_player.player_key
fact_nextgen_passing.team_key        → dim_team.team_key
fact_nextgen_passing.season_week_key → dim_season_week.season_week_key

fact_nextgen_rushing.player_key      → dim_player.player_key
fact_nextgen_rushing.team_key        → dim_team.team_key
fact_nextgen_rushing.season_week_key → dim_season_week.season_week_key

fact_nextgen_receiving.player_key      → dim_player.player_key
fact_nextgen_receiving.team_key        → dim_team.team_key
fact_nextgen_receiving.season_week_key → dim_season_week.season_week_key
```

### Example Cross-Domain Queries

**Cap efficiency vs. EPA:**
```sql
SELECT p.display_name, c.cap_number, c.position,
       SUM(pp.epa) AS total_epa, COUNT(*) AS plays
FROM gold.fact_player_contract c
JOIN gold.dim_player p ON c.player_key = p.player_key
JOIN gold.fact_pass_play pp ON c.player_key = pp.passer_player_key AND c.season = pp.season
WHERE c.season = 2024
GROUP BY p.display_name, c.cap_number, c.position
```

**Combine athleticism vs. Next Gen Stats separation:**
```sql
SELECT cb.player_name, cb.forty_yard_dash, cb.vertical_jump,
       AVG(nr.avg_separation) AS avg_separation
FROM gold.fact_combine cb
JOIN gold.fact_nextgen_receiving nr ON cb.player_key = nr.player_key
WHERE cb.position = 'WR'
GROUP BY cb.player_name, cb.forty_yard_dash, cb.vertical_jump
```